In [1]:
# ==============================================================================
# NOTEBOOK: 00_silver_pipeline_and_audit
# DESCRIPCIÓN: Pipeline Silver (Promoción Bronze -> Silver) + Auditoría
# ==============================================================================
from pyspark.sql import functions as F
from tfm_mobility.processors.silver_processor import SilverProcessor

print("================================================================================")
print("🚀 INICIANDO CANALIZACIÓN DE DATOS SILVER, AUDITORÍA Y CALIDAD DE DATOS")
print("================================================================================\n")

# 1. EJECUCIÓN DEL PIPELINE SILVER
print("--- 1. PROCESANDO TABLAS BRONZE A SILVER ---")
try:
    processor = SilverProcessor(spark)
    processor.run_all_silver_pipeline()
    print("✅ Transformaciones a Silver completadas con éxito.\n")
except Exception as e:
    print(f"❌ Error ejecutando el pipeline de Silver: {e}\n")

# 2. INVENTARIO DE TABLAS DELTA SILVER
print("="*80)
print("📦 RESUMEN DE VOLUMETRÍA EN TABLAS DELTA SILVER")
print("="*80)

silver_tables = ["silver_weather", "silver_nasa_fires", "silver_dgt_traffic", "silver_osm_roads"]

for t_name in silver_tables:
    try:
        cnt = spark.table(t_name).count()
        print(f"   • {t_name:<25}: {cnt:,} registros Delta limpios.")
    except Exception:
        print(f"   • {t_name:<25}: ❌ No encontrada en catálogo.")

# 3. AUDITORÍA DE CALIDAD Y COMPLETITUD EN SILVER
print("\n" + "="*80)
print("📊 AUDITORÍA DE ESTRUCTURA Y COMPLETITUD EN CAPA SILVER")
print("="*80 + "\n")

def run_silver_audit():
    for table_name in silver_tables:
        print("-" * 80)
        print(f"🔎 AUDITANDO TABLA SILVER: {table_name.upper()}")
        print("-" * 80)
        try:
            df = spark.table(table_name)
            total_records = df.count()
            print(f"🔹 Nº Total de Registros: {total_records:,} | Total Columnas: {len(df.columns)}")

            if total_records == 0:
                continue

            print("\n🔑 Completitud por Columna:")
            for col_name, dtype in df.dtypes:
                n_cnt = df.filter(F.col(col_name).isNull()).count()
                completeness = round(((total_records - n_cnt) / total_records) * 100, 2)
                print(f"   • {col_name:<30} ({dtype:<10}): Completitud: {completeness:>6.2f}%")

            print("\n📄 Muestra de Registros Transformados:")
            df.show(3, truncate=40)
            print("\n")
        except Exception as e:
            print(f"❌ Error auditando la tabla {table_name}: {e}\n")

run_silver_audit()

StatementMeta(, 030baff5-265c-429c-9770-76daa253de51, 5, Finished, Available, Finished, False)

2026-09-02 16:06:20,880 - INFO - 🚀 [SILVER] Iniciando pipeline de transformación de capa Bronze a Silver...
2026-09-02 16:06:47,993 - INFO - ✨ Tabla Silver Delta 'silver_weather' creada por primera vez.
2026-09-02 16:07:08,516 - INFO - ✅ Tabla Silver Delta 'silver_nasa_fires' actualizada mediante MERGE.
2026-09-02 16:08:00,462 - INFO - ✅ Tabla Silver Delta 'silver_dgt_traffic' actualizada mediante MERGE.


🚀 INICIANDO CANALIZACIÓN DE DATOS SILVER, AUDITORÍA Y CALIDAD DE DATOS

--- 1. PROCESANDO TABLAS BRONZE A SILVER ---
✅ Transformaciones a Silver completadas con éxito.

📦 RESUMEN DE VOLUMETRÍA EN TABLAS DELTA SILVER
   • silver_weather           : 1,150,848 registros Delta limpios.
   • silver_nasa_fires        : 6,407 registros Delta limpios.
   • silver_dgt_traffic       : 154,166 registros Delta limpios.
   • silver_osm_roads         : 5,000 registros Delta limpios.

📊 AUDITORÍA DE ESTRUCTURA Y COMPLETITUD EN CAPA SILVER

--------------------------------------------------------------------------------
🔎 AUDITANDO TABLA SILVER: SILVER_WEATHER
--------------------------------------------------------------------------------
🔹 Nº Total de Registros: 1,150,848 | Total Columnas: 17

🔑 Completitud por Columna:
   • latitude                       (double    ): Completitud: 100.00%
   • longitude                      (double    ): Completitud: 100.00%
   • elevation                      (dou